In [94]:
## Importing Libraries
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import numpy as np
import pickle

In [95]:
### Load the trained model, scaler pickle file, onehot encoding
model = load_model('model.h5')

## Load the encoder and scaler
with open ('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open ('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open ('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [96]:
## Example input data
input_data = {
    'CreditScore' : 600,
    'Geography' : 'France',
    'Gender' : 'Male',
    'Age' : 40,
    'Tenure' : 3,
    'Balance' : 60000,
    'NumOfProducts' : 2,
    'HasCrCard' : 1,
    'IsActiveMember' :1,
    'EstimatedSalary' :50000
}

In [97]:
input_df=pd.DataFrame([input_data])


In [98]:
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

e:\PG BITS\Projects\ANN Classification\myvenv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [99]:
## Combine one-hot encoded columns with input data
input_data = pd.concat([input_df.reset_index(drop=True),geo_encoded_df],axis=1)
input_data


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [100]:
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])

In [101]:
input_df = pd.concat([input_df.drop('Geography',axis=1),geo_encoded_df],axis=1)


In [102]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [104]:
input_scaled =scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [105]:
## Predict Churn
prediction=model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 159ms/step


array([[0.02764264]], dtype=float32)

In [111]:
prediction_prob = prediction[0][0]
prediction_prob

0.027642645

In [109]:
if prediction_prob  > 0.5:
    print("Customer is likely to churn")
else:
    print("Customer is not likely to churn") 


Customer is not likely to churn
